# TF-IDF + Logistic Regression + sublinear_tf=True

Experimento G - redução de dimensionalidade

In [1]:
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

## 1. Carregamento dos dados

In [2]:
df = pd.read_csv("train_limpo.csv")

Y = df["clarity"]

print(df.shape)
df.head()

(20092, 2)


,resp_text,clarity
0,prezado a senhor a esclarecemos que o serviço ...,c5
1,prezada cidadã as informações sobre óbitos no ...,c1
2,prezado senhor julio a ouvidoria geral da petr...,c1
3,prezado a senhor a esclarecemos que o serviço ...,c234
4,senhor o serviço de informações ao cidadão do ...,c234


## 2. Separação dos dados

In [3]:
x_train_text, x_test_text, y_train, y_test = train_test_split(
    df["resp_text"],
    Y,
    test_size=0.2,
    random_state=123,
    stratify=Y
)

print("Treino:", x_train_text.shape)
print("Teste:", x_test_text.shape)

Treino: (16073,)
Teste: (4019,)


## 3. Vetorização TF-IDF

Neste experimento, utilizamos `sublinear_tf=True`, que aplica uma ponderação sublinear à frequência dos termos.

In [ ]:
vect = TfidfVectorizer(
    analyzer="word",
    sublinear_tf=True,
    ngram_range=(1, 3)
)

x_train = vect.fit_transform(x_train_text)
x_test = vect.transform(x_test_text)

print("Treino TF-IDF:", x_train.shape)
print("Teste TF-IDF:", x_test.shape)

Treino TF-IDF: (16073, 1196293)
Teste TF-IDF: (4019, 1196293)


## 4. Regressão Logística

In [ ]:
clf = LogisticRegression(
    C=1,
    class_weight="balanced",
    max_iter=1000
)

clf.fit(
    x_train,
    y_train
)

LogisticRegression(C=1, class_weight='balanced', max_iter=1000)

## 5. Avaliação

In [ ]:
predicted = clf.predict(x_test)

score = f1_score(
    y_test,
    predicted,
    average="macro"
)

accuracy = accuracy_score(
    y_test,
    predicted
)

print(f"Macro F1: {score:.4f}")
print(f"Accuracy: {accuracy:.4f}")

Macro F1: 0.4799
Accuracy: 0.4800


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        predicted
    )
)

              precision    recall  f1-score   support

          c1       0.51      0.52      0.51      1269
        c234       0.41      0.39      0.40      1371
          c5       0.52      0.53      0.52      1379

    accuracy                           0.48      4019
   macro avg       0.48      0.48      0.48      4019
weighted avg       0.48      0.48      0.48      4019



## Experimento G — TF-IDF + redução de dimensionalidade

Neste experimento, utilizamos Truncated SVD para reduzir a dimensionalidade da matriz TF-IDF antes da classificação com Regressão Logística.

Mantemos a configuração do Experimento A e testamos diferentes quantidades de componentes.

In [4]:
# Verificar a quantidade de características antes da redução

tfidf = TfidfVectorizer(
    analyzer="word",
    sublinear_tf=True,
    ngram_range=(1, 3),
    dtype="float32"
)

X_tfidf = tfidf.fit_transform(x_train_text)

print("Dimensões antes da redução:", X_tfidf.shape)
print("Quantidade de características:", X_tfidf.shape[1])

/usr/local/lib/python3.13/dist-packages/sklearn/feature_extraction/text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


Dimensões antes da redução: (16073, 1196293)
Quantidade de características: 1196293


In [ ]:
componentes = [100, 200, 300, 400, 500, 700, 900]

In [ ]:
componentes = [1100, 1300, 1500]

In [5]:
componentes = [1700, 1900, 2000]

In [6]:
def criar_pipeline(n_componentes):
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="word",
            sublinear_tf=True,
            ngram_range=(1, 3),
            dtype="float32",
            max_features=500000

        )),
        ("svd", TruncatedSVD(
            n_components=n_componentes,
            n_iter=5,
            random_state=123
        )),
        ("logreg", LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=1000
        ))
    ])

    return pipeline

In [ ]:
# ============================================================
# AVALIAÇÃO INICIAL — EXPERIMENTO G
# ============================================================

for n in componentes:

    print("\n" + "=" * 45)
    print(f"EXPERIMENTO G — {n} COMPONENTES")
    print("=" * 45)

    # Criação do pipeline
    pipeline = criar_pipeline(n)

    # Treinamento
    pipeline.fit(x_train_text, y_train)

    # Predições
    predicted = pipeline.predict(x_test_text)

    # Métricas
    score = f1_score(
        y_test,
        predicted,
        average="macro"
    )

    accuracy = accuracy_score(
        y_test,
        predicted
    )

    print(f"Macro F1: {score:.4f}")
    print(f"Accuracy: {accuracy:.4f}")


EXPERIMENTO G — 1700 COMPONENTES
Macro F1: 0.4740
Accuracy: 0.4745

EXPERIMENTO G — 1900 COMPONENTES


/usr/local/lib/python3.13/dist-packages/sklearn/feature_extraction/text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


## Validação Cruzada (não rodei)

In [ ]:
X = df["resp_text"]
y = Y

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

In [ ]:
# ============================================================
# VALIDAÇÃO CRUZADA — EXPERIMENTO G
# ============================================================

for n in componentes:

    print("\n" + "=" * 45)
    print(f"VALIDAÇÃO CRUZADA — {n} COMPONENTES")
    print("=" * 45)

    pipeline = criar_pipeline(n)

    results = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring={
            "f1_macro": "f1_macro",
            "accuracy": "accuracy"
        },
        return_train_score=True,
        n_jobs=1
    )

    # Resultados por fold
    print("\nResultados por fold:\n")

    for i in range(5):
        print(
            f"Fold {i+1}: "
            f"F1 Macro = {results['test_f1_macro'][i]:.4f} | "
            f"Accuracy = {results['test_accuracy'][i]:.4f}"
        )

    # Médias e desvios padrão
    print("\nResultado da validação cruzada:")

    print(
        f"Macro F1: "
        f"{results['test_f1_macro'].mean():.4f} "
        f"± {results['test_f1_macro'].std():.4f}"
    )

    print(
        f"Accuracy: "
        f"{results['test_accuracy'].mean():.4f} "
        f"± {results['test_accuracy'].std():.4f}"
    )

    # Resultados de treino
    print("\nTreino:")

    print(
        f"Macro F1: "
        f"{results['train_f1_macro'].mean():.4f} "
        f"± {results['train_f1_macro'].std():.4f}"
    )

    print(
        f"Accuracy: "
        f"{results['train_accuracy'].mean():.4f} "
        f"± {results['train_accuracy'].std():.4f}"
    )